In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("models/final_model")
model = AutoModelForSequenceClassification.from_pretrained("models/final_model")

model.to(device)
model.eval()


def predict(text, context=""):
    combined = context + " [SEP] " + text

    inputs = tokenizer(
        combined,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    sarcasm_prob = probs[0][1].item()

    if sarcasm_prob > 0.6:
        label = "Sarcastic 😏"
    elif sarcasm_prob < 0.4:
        label = "Not Sarcastic 🙂"
    else:
        label = "Uncertain 🤔"

    return {"label": label, "confidence": round(sarcasm_prob, 3)}

# TEST
if __name__ == "__main__":
    print(predict("1. Oh great, another bug in my code"))
    print(predict("2. I am going to the store"))
    print(predict("3. Area man absolutely thrilled to work overtime again"))
    print(predict("4. Local man wins lottery for third time this week"))
    print(predict("5. Area man thrilled to pay taxes again"))
    print(predict("6. Scientists discover water on Mars"))
    print(predict(
        "\nOh great, another bug",
        context="I have been debugging for 5 hours"
    ))


{'label': 'Not Sarcastic 🙂', 'confidence': 0.013}
{'label': 'Not Sarcastic 🙂', 'confidence': 0.005}
{'label': 'Sarcastic 😏', 'confidence': 0.998}
{'label': 'Sarcastic 😏', 'confidence': 0.927}
{'label': 'Sarcastic 😏', 'confidence': 0.994}
{'label': 'Not Sarcastic 🙂', 'confidence': 0.003}
{'label': 'Uncertain 🤔', 'confidence': 0.524}
